# Select target basins 

In [ ]:
# general
import pandas as pd
import geopandas as gpd

from src.utils import load_config
cfg = load_config()

# Data

In [ ]:
# HydroATLAS level 08
basins = gpd.read_file(cfg["dir"]["data"] + "/hydroatlas/hybas_sa_lev08_v1c.shp")
basins.columns = [col.lower() for col in basins.columns]
basins = basins[["hybas_id", "next_down", "next_sink", "main_bas", "dist_sink", "dist_main", "sub_area", "up_area", "geometry"]]
basins = basins.rename(columns={"hybas_id": "gauge_id"})

# RGI v6.0 - Andes
glaciers = pd.concat([gpd.read_file(cfg["dir"]["data"]  + "/glacier_outlines/RGI60_16.shp"), 
                      gpd.read_file(cfg["dir"]["data"]  + "/glacier_outlines/RGI60_17.shp")])
glaciers = glaciers[(glaciers.CenLon < cfg["area"]["max_lon"]) & 
                    (glaciers.CenLon > cfg["area"]["min_lon"]) & 
                    (glaciers.CenLat < cfg["area"]["max_lat"])]
glaciers = glaciers.rename(columns={"RGIId": "rgi_id"})
glaciers = glaciers.union_all()

## Filtering based on glacier area in %

In [ ]:
to_remove = [6080838340, 6080840550, 6080836740]

# select all basins with glaciers
basins_with_glaciers = basins[basins.intersects(glaciers)].copy()
# glacier cover in % of basin_area, both areas in UTM 19S (EPSG:32719) as in AndeanGC v1.2
glacier_area = basins_with_glaciers.intersection(glaciers).to_crs(epsg=32719).area * 1e-6
basin_area   = basins_with_glaciers.to_crs(epsg=32719).area * 1e-6
basins_with_glaciers['glacier_cover_RGI60'] = glacier_area * 100 / basin_area
basins_with_glaciers = basins_with_glaciers[basins_with_glaciers.glacier_cover_RGI60 > 0]
basins_with_glaciers = basins_with_glaciers[~basins_with_glaciers.gauge_id.isin(to_remove)]

print(f"Number of basins with glaciers: {len(basins_with_glaciers)}")

In [ ]:
# add key basins with high water demand but no glaciers (e.g., Santiago, Mendoza)
to_add = [6080891940, 6080892210, 6080878030, 6080878640, 6080884520, 
          6080898850, 6080905100, 6080902210, 6080905130, 6080895720, 
          6080905930, 6080892180, 6080910100, 6080916390,
          6080921600, 6080906070, 6080882150, 6080027290]
basins_to_add = basins[basins.gauge_id.isin(to_add)]
basins_to_recover = pd.concat([basins_with_glaciers, basins_to_add])

# add upstream basins to model the full upstream nework
basins_to_recover = basins[basins.next_down.isin(basins_to_recover.gauge_id)] 

target_basins = pd.concat([basins_with_glaciers, basins_to_recover, basins_to_add])
target_basins = target_basins.drop_duplicates(subset="gauge_id") 

for _ in range(30):
    basins_to_recover = basins[basins.next_down.isin(target_basins.gauge_id)]
    target_basins = pd.concat([target_basins, basins_to_recover])
    target_basins = target_basins.drop_duplicates(subset="gauge_id") 

target_basins = target_basins.fillna(0) # glacier area = 0% for these basins
target_basins["gauge_id"] = target_basins.gauge_id.astype(str) # ensure gauge_id is string type
target_basins = target_basins.set_index("gauge_id")
target_basins.index.name = "gauge_id"
target_basins.to_file("data/hydro/basins_hydroATLAS_lev08.gpkg")

print(f"Number of total basins (including upstream and downstream): {len(target_basins)}")